# Credit Scoring with Explainability — Statlog German Credit Dataset

**What this is:** an auditable credit scorecard and the decision policy that sits on top of
it, built so a risk committee could sign it off — plus an honest account of what this
dataset can and cannot support.

**Scope statement, up front.** Statlog German Credit is 1,000 rows, 300 bads, collected in
1994, with no time dimension and no vintage structure. The published ceiling on this
dataset is roughly **AUC 0.78–0.80**, and essentially every modelling approach lands
there. Chasing another 0.005 of AUC is not where the value is. The work that actually
changes decisions is:

1. **A loss function denominated in money, not counts.** Expected loss is
   `P(default) x EAD x LGD`. A missed default on an 18,000 DM loan is not the same event
   as one on a 500 DM loan, so a single flat cost ratio cannot produce the right cutoff.
2. **A feature set that is legal to use.** Sex and foreign-worker status are prohibited
   bases under ECOA / Reg B. They are excluded from the model here and retained only as
   attributes for measuring disparity.
3. **Uncertainty stated honestly.** On a 250-row test set the standard error on AUC is
   around +/- 0.035. Point estimates quoted to four decimals are false precision.
4. **One artifact, one model card.** The notebook persists exactly the object the serving
   layer loads, so the documentation cannot drift away from what is in production.

**Deliberately not included:** hyperparameter search, SMOTE, stacked ensembles, and extra
challenger families. At n=750 training rows with ~45 bads per validation fold, the
selection noise of a tuning run exceeds any real effect it could find — a 25-trial search
will beat its own mean by roughly 0.04 AUC through selection alone, with no genuine gain.
Section 5 uses fixed, defensible regularisation instead and explains why.

**Pipeline:** EDA -> supervised WOE binning -> sign-constrained logistic scorecard ->
probability calibration -> regularised LightGBM challenger -> significance testing ->
exposure-weighted decision policy -> score bands & stability -> explainability ->
fairness -> model card -> serialised artifact.

In [ ]:
import os
import json
import urllib.request
import warnings
from dataclasses import dataclass, asdict
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.optimize import minimize
from scipy.stats import norm

import sklearn
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss, confusion_matrix,
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
    cross_val_score, cross_val_predict,
)
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import shap

# Model components live in scorecard.py, not in these cells. A joblib artifact pickles a
# reference to the defining module, so anything defined in a notebook cell is saved as
# __main__.X and cannot be loaded by the Flask process. Importing them here is what lets
# the notebook and the serving layer share one definition instead of two copies.
from scorecard import (
    WOETransformer, SignConstrainedLogit, ScorecardModel, BandedPolicy, expected_loss,
)

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')

## 0. Configuration

Every assumption in the notebook lives here, including the two that drive the money-based
loss function. `lgd` and `annual_margin` are **assumptions, not measurements** — this
dataset carries no recovery or pricing data. They are set to conventional unsecured
consumer-lending values and Section 6 shows how sensitive the chosen cutoff is to them,
which is the honest way to use a number you had to assume.

In [ ]:
@dataclass(frozen=True)
class Config:
    random_state: int = 42
    test_size: float = 0.25

    # cross-validation
    n_splits: int = 5
    n_repeats: int = 5          # RepeatedStratifiedKFold: 25 fits, tightens the CV estimate

    # WOE binning
    n_bins: int = 5             # max bins per numeric feature (supervised, tree-derived)
    min_bin_frac: float = 0.05  # no bin smaller than 5% of the training fold
    min_iv: float = 0.02        # information-value cutoff for keeping a feature

    # money-denominated loss (assumptions — see sensitivity analysis in Section 6)
    lgd: float = 0.75           # loss given default, share of exposure lost
    annual_margin: float = 0.045  # net annual margin forgone on a wrongly declined good loan

    # legacy flat-cost weights, retained ONLY to reproduce the old policy for comparison
    legacy_cost_fn: float = 5.0
    legacy_cost_fp: float = 1.0

    # PDO scorecard scaling
    pdo: float = 20.0
    base_score: float = 600.0
    base_odds: float = 50.0
    score_floor: float = 300.0
    score_cap: float = 850.0

    # inference / reporting
    n_boot: int = 2000          # bootstrap resamples for metric confidence intervals
    n_amount_bands: int = 3     # exposure bands for the banded decision policy
    n_score_bands: int = 10     # score-band risk table granularity

CFG = Config()

# Prohibited bases under ECOA / Reg B. Kept in the dataframe for fairness measurement,
# never passed to a model, never surfaced in an adverse-action reason code.
PROTECTED_FEATURES = ['personal_status_sex', 'foreign_worker']

# `age` is also a protected basis in the US, but is permitted in several other
# jurisdictions and is retained here under an explicit monotonic constraint (older ->
# non-increasing risk) so its effect stays inspectable and cannot invert. Section 8
# measures age disparity regardless. Moving it into PROTECTED_FEATURES is a one-line
# change if the deployment jurisdiction requires it.

## 1. Data loading

The original **Statlog (German Credit Data)** set: 1,000 applicants, 20 attributes, coded
categorical values (`A11`, `A12`, ...), binary good/bad target. Not the simplified Kaggle
"Hofmann" CSV, which drops attributes and has no target column.

The loader prefers a local copy and falls back to a mirror so the notebook runs
standalone. Category decoding is table-driven via `CATEGORY_MAPS`.

In [ ]:
COLUMNS = [
    'checking_status', 'duration_months', 'credit_history', 'purpose',
    'credit_amount', 'savings_status', 'employment_since', 'installment_rate',
    'personal_status_sex', 'other_debtors', 'residence_since', 'property',
    'age', 'other_installment_plans', 'housing', 'existing_credits',
    'job', 'num_dependents', 'telephone', 'foreign_worker', 'target_raw',
]

DATA_CANDIDATES = ['german.data', 'german_statlog.csv', 'german_credit_statlog.csv']
DATA_URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/german.csv'

CATEGORY_MAPS = {
    'checking_status': {'A11': '< 0 DM', 'A12': '0-200 DM', 'A13': '>= 200 DM', 'A14': 'no checking account'},
    'credit_history': {
        'A30': 'no credits taken / all paid duly', 'A31': 'all credits at this bank paid duly',
        'A32': 'existing credits paid duly till now', 'A33': 'delay in paying off in the past',
        'A34': 'critical account / other credits elsewhere',
    },
    'purpose': {
        'A40': 'car (new)', 'A41': 'car (used)', 'A42': 'furniture/equipment',
        'A43': 'radio/television', 'A44': 'domestic appliances', 'A45': 'repairs',
        'A46': 'education', 'A47': 'vacation', 'A48': 'retraining',
        'A49': 'business', 'A410': 'others',
    },
    'savings_status': {
        'A61': '< 100 DM', 'A62': '100-500 DM', 'A63': '500-1000 DM',
        'A64': '>= 1000 DM', 'A65': 'unknown/no savings account',
    },
    'employment_since': {
        'A71': 'unemployed', 'A72': '< 1 year', 'A73': '1-4 years',
        'A74': '4-7 years', 'A75': '>= 7 years',
    },
    'personal_status_sex': {
        'A91': 'male: divorced/separated', 'A92': 'female: divorced/separated/married',
        'A93': 'male: single', 'A94': 'male: married/widowed', 'A95': 'female: single',
    },
    'other_debtors': {'A101': 'none', 'A102': 'co-applicant', 'A103': 'guarantor'},
    'property': {
        'A121': 'real estate', 'A122': 'building society savings/life insurance',
        'A123': 'car or other property', 'A124': 'unknown/no property',
    },
    'other_installment_plans': {'A141': 'bank', 'A142': 'stores', 'A143': 'none'},
    'housing': {'A151': 'rent', 'A152': 'own', 'A153': 'for free'},
    'job': {
        'A171': 'unemployed/unskilled non-resident', 'A172': 'unskilled resident',
        'A173': 'skilled employee/official', 'A174': 'management/self-employed/highly qualified',
    },
    'telephone': {'A191': 'none', 'A192': 'yes, registered'},
    'foreign_worker': {'A201': 'yes', 'A202': 'no'},
}

ALL_NUMERIC = ['duration_months', 'credit_amount', 'age', 'installment_rate',
               'residence_since', 'existing_credits', 'num_dependents']
ALL_CATEGORICAL = list(CATEGORY_MAPS.keys())


def load_data() -> pd.DataFrame:
    '''Load the raw Statlog file, recode the target, decode category codes to labels.'''
    data_path = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)
    if data_path is None:
        print(f'No local copy found - downloading from {DATA_URL} ...')
        urllib.request.urlretrieve(DATA_URL, DATA_CANDIDATES[1])
        data_path = DATA_CANDIDATES[1]
    print('Using:', data_path)

    # sep=None + engine='python' auto-detects whitespace (UCI .data) or comma (mirror)
    raw = pd.read_csv(data_path, header=None, names=COLUMNS, sep=None, engine='python')

    # target_raw: 1 = Good, 2 = Bad (UCI convention) -> 0 = Good, 1 = Bad
    raw['target'] = raw['target_raw'].map({1: 0, 2: 1})
    assert raw['target'].isnull().sum() == 0, 'Unexpected target code found'

    df = raw.drop(columns=['target_raw']).copy()
    for col, mapping in CATEGORY_MAPS.items():
        df[col] = df[col].map(mapping)
    assert df.drop(columns=['target']).isnull().sum().sum() == 0, 'Unmapped codes remain!'
    return df


df = load_data()
print(df.shape)
print(df['target'].value_counts(normalize=True).rename({0: 'Good', 1: 'Bad'}).rename('share'))
df.head()

## 2. EDA

In [ ]:
target_share = df['target'].value_counts(normalize=True).rename({0: 'Good', 1: 'Bad'})

fig, ax = plt.subplots(figsize=(4, 4))
target_share.plot(kind='bar', color=['#3B76AF', '#E1584B'], ax=ax)
ax.set_ylabel('Share of applicants')
ax.set_title('Class balance: Good vs Bad')
plt.tight_layout()
plt.show()

In [ ]:
for col in ['duration_months', 'credit_amount', 'age']:
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.histplot(data=df, x=col, hue='target', kde=True, ax=ax,
                 palette={0: '#3B76AF', 1: '#E1584B'}, bins=30, alpha=0.6)
    ax.set_title(col)
    plt.tight_layout()
    plt.show()

In [ ]:
for col in ['checking_status', 'credit_history', 'purpose']:
    ct = pd.crosstab(df[col], df['target'], normalize='index').rename(
        columns={0: 'good_rate', 1: 'bad_rate'})
    print(f'\n--- {col} vs target (row-normalized) ---')
    print(ct.sort_values('bad_rate', ascending=False))

In [ ]:
corr = df[ALL_NUMERIC].corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation among numeric features')
plt.tight_layout()
plt.show()

**Reading the EDA:** applicants with no checking account or a strongly negative one, a
critical/no-credit-history record, and higher-risk purposes show visibly higher bad rates.
Numeric features are only weakly correlated (no severe multicollinearity going into
modelling); `duration_months` and `credit_amount` move together, as expected — longer
loans are larger loans. That pairing matters in Section 6: exposure and term jointly
drive the money value of a decision error.

## 3. Feature set: what the model is allowed to see

`personal_status_sex` and `foreign_worker` are **prohibited bases** under ECOA / Reg B and
are removed from the modelling matrix here. They stay in `df` so Section 8 can measure
disparity against them — you cannot audit for bias on an attribute you have thrown away.

This is not only a compliance point, it closes a live defect: the previously deployed
model consumed `personal_status_sex`, and its Tree SHAP reason codes could therefore emit
a literal adverse-action line of the form
`personal_status_sex = female: divorced/separated/married (contributed +0.14 to risk)`.
That is a prohibited basis printed on the exact document ECOA governs.

Removing features cannot improve fit, so the relevant question is what it costs. Section 5
reports the AUC delta against a variant that keeps them, which is the number a credit
committee will ask for. Dropping the attributes does not remove proxy effects, which is
why measurement in Section 8 continues either way.

In [ ]:
MODEL_FEATURES = [c for c in df.columns if c not in ['target'] + PROTECTED_FEATURES]
NUMERIC_FEATS = [c for c in ALL_NUMERIC if c in MODEL_FEATURES]
CATEGORICAL_FEATS = [c for c in ALL_CATEGORICAL if c in MODEL_FEATURES]

X = df[MODEL_FEATURES]
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CFG.test_size, random_state=CFG.random_state, stratify=y
)

# Held back purely for the fairness audit — never joined to the modelling matrix.
audit_train = df.loc[X_train.index, PROTECTED_FEATURES + ['age', 'credit_amount']]
audit_test = df.loc[X_test.index, PROTECTED_FEATURES + ['age', 'credit_amount']]

cv = RepeatedStratifiedKFold(n_splits=CFG.n_splits, n_repeats=CFG.n_repeats,
                             random_state=CFG.random_state)
cv_single = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True,
                            random_state=CFG.random_state)  # for out-of-fold predictions

print(f'Modelling on {len(MODEL_FEATURES)} features; excluded as prohibited bases: {PROTECTED_FEATURES}')
print(f'Train {X_train.shape}, test {X_test.shape}')

## 4. Scorecard: supervised WOE binning + sign-constrained logistic regression

Two changes from a textbook WOE scorecard, both fixing real defects:

**Supervised binning.** Equal-frequency `qcut` bins ignore the target and, on 750 rows,
routinely produce sparse non-monotonic bins whose WOE is mostly noise. Bins here come from
a shallow decision tree fit on the feature against the target, with `min_samples_leaf`
enforcing a floor of 5% of the fold, then a merge pass that enforces monotonic WOE. This
is what removes the instability that caused the sign flips below.

**Sign constraints.** WOE is constructed so that a higher value always means a safer bin,
so on `log-odds(bad)` *every* coefficient must be negative. An unconstrained fit on
correlated WOE features reliably flips one or two signs, which means the scorecard would
award points for being riskier — not shippable, and the previous version of this notebook
documented the problem and shipped anyway. `SignConstrainedLogit` imposes the bound
directly in the optimiser, so the constraint holds by construction rather than by luck.

The transformer fits bins, WOE maps and IV selection inside `fit`, so all of it is
re-derived per fold when used with `cross_val_score` — no leakage across the split.

In [ ]:
def make_scorecard_pipeline(numeric_feats, categorical_feats) -> Pipeline:
    '''WOE encoding + sign-constrained logistic regression, as one CV-safe estimator.

    Both stages are defined in scorecard.py. Wrapping them in a Pipeline is what enforces
    "fit the bins on training rows only" automatically on every fold, rather than relying
    on remembering to split before encoding.
    '''
    return Pipeline([
        ('woe', WOETransformer(numeric_feats, categorical_feats, n_bins=CFG.n_bins,
                               min_iv=CFG.min_iv, min_bin_frac=CFG.min_bin_frac,
                               random_state=CFG.random_state)),
        ('clf', SignConstrainedLogit(C=1.0, sign=-1)),
    ])

In [ ]:
scorecard_pipe = make_scorecard_pipeline(NUMERIC_FEATS, CATEGORICAL_FEATS)

cv_aucs = cross_val_score(scorecard_pipe, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'Scorecard {CFG.n_splits}-fold x{CFG.n_repeats} repeated CV AUC: '
      f'{cv_aucs.mean():.3f} +/- {cv_aucs.std():.3f}  (n={len(cv_aucs)} fits)')

scorecard_pipe.fit(X_train, y_train)
woe_step = scorecard_pipe.named_steps['woe']
lr = scorecard_pipe.named_steps['clf']

iv_table = woe_step.iv_summary_.to_frame('IV')
iv_table['kept'] = iv_table.index.isin(woe_step.keep_feats_)
print(f'\nKept {len(woe_step.keep_feats_)} of {len(woe_step.iv_summary_)} features '
      f'(IV >= {CFG.min_iv}, fit on train only)')
iv_table

**On `checking_status` (highest IV by a wide margin):** this is a genuinely strong,
well-documented signal — whether an applicant holds a checking account at all, and its
balance, is a classic early indicator of financial stability. It is not target leakage,
but an IV this high always deserves a second look in a real deployment before it is
trusted. IV is reported once, here, from the fitted transformer; there is no separate
full-data IV pass, because computing it twice invites the two numbers to disagree.

In [ ]:
coef_table = pd.DataFrame({
    'feature': woe_step.feature_names_out_,
    'coefficient': lr.coef_[0],
}).sort_values('coefficient')

n_wrong_sign = int((coef_table['coefficient'] > 1e-9).sum())
print(f'Coefficients with the wrong sign: {n_wrong_sign} (must be 0)')
print(f'Optimiser converged: {lr.converged_}')
print(f'Coefficients pinned exactly at the constraint boundary: '
      f'{int(np.isclose(coef_table["coefficient"], 0.0).sum())} '
      f'(these are features the constrained fit found no use for)')
coef_table

## 5. Calibration

The decision policy in Section 7 compares `P(default)` against a threshold, and the
scorecard converts `P(default)` into points. Both are meaningless unless the probability
is *calibrated* — a model can rank perfectly (high AUC) while being systematically over-
or under-confident, and a threshold read off an uncalibrated score points at the wrong
applicants.

Calibration here is **Platt scaling fit on out-of-fold decision values**: not on the
training fit (optimistic) and not on the test set (leakage). It is deliberately a plain
two-parameter sigmoid rather than isotonic regression — isotonic needs far more than 750
rows and would overfit the calibration curve itself.

A sigmoid on a linear model stays linear in log-odds (`z_cal = a*z + b`), so calibration
is just a rescaling of the coefficients. That keeps the scorecard **exactly additive**:
per-feature points sum to the total score with no approximation anywhere between the
model and the reason codes.

In [ ]:
oof_z = cross_val_predict(scorecard_pipe, X_train, y_train, cv=cv_single,
                          method='decision_function')

platt = LogisticRegression(C=1e6, solver='lbfgs')
platt.fit(oof_z.reshape(-1, 1), y_train)
cal_a = float(platt.coef_[0][0])
cal_b = float(platt.intercept_[0])

FACTOR = CFG.pdo / np.log(2)
OFFSET = CFG.base_score - FACTOR * np.log(CFG.base_odds)

# The deployed object. Everything downstream — points, reason codes, the API — goes
# through this one instance, so the notebook exercises exactly what gets serialised.
scorecard = ScorecardModel(
    woe=woe_step,
    cal_coef=pd.Series(cal_a * lr.coef_[0], index=woe_step.feature_names_out_),
    cal_intercept=cal_a * float(lr.intercept_[0]) + cal_b,
    contrib_mean=(woe_step.transform(X_train)
                  * (cal_a * lr.coef_[0])).mean(),
    pdo={'factor': FACTOR, 'offset': OFFSET,
         'floor': CFG.score_floor, 'cap': CFG.score_cap},
    feature_labels={},          # filled in Section 9
    protected_features=PROTECTED_FEATURES,
)

oof_proba_sc = 1.0 / (1.0 + np.exp(-(cal_a * oof_z + cal_b)))
oof_proba_raw = 1.0 / (1.0 + np.exp(-oof_z))

print(f'Platt scaling on out-of-fold values: a={cal_a:.4f}, b={cal_b:.4f}')
print('(a < 1 means the raw model was over-confident and has been shrunk toward the base rate)')
print(f'\nBrier score, out-of-fold — raw: {brier_score_loss(y_train, oof_proba_raw):.4f}'
      f'   calibrated: {brier_score_loss(y_train, oof_proba_sc):.4f}   (lower is better)')

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for label, proba in [('raw', oof_proba_raw), ('calibrated', oof_proba_sc)]:
    frac_pos, mean_pred = calibration_curve(y_train, proba, n_bins=8, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', label=label)
ax.plot([0, 1], [0, 1], '--', color='grey', linewidth=1, label='perfect')
ax.set_xlabel('Mean predicted P(bad)')
ax.set_ylabel('Observed bad rate')
ax.set_title('Reliability diagram (out-of-fold, training set)')
ax.legend()
plt.tight_layout()
plt.show()

### Points-based scorecard (PDO scaling)

```
factor = PDO / ln(2)
offset = base_score - factor * ln(base_odds)
score  = offset - factor * (calibrated log-odds of bad)
```

The intercept is split evenly across features so each `points_i` sums exactly back to the
total. The old notebook checked this identity and called it a sanity check — it was a
tautology that could only ever report float noise. The check below is the one that can
actually fail: whether every coefficient respects the sign constraint, and how many rows
hit the display clip.

In [ ]:
train_scores = scorecard.points(X_train)
test_scores = scorecard.points(X_test)

print('Train score distribution:')
print(train_scores.describe().round(1))

unclipped = scorecard.points_frame(X_train).sum(axis=1)
print(f'\nCoefficients violating the sign constraint: '
      f'{int((scorecard.cal_coef > 1e-9).sum())} (must be 0)')
print(f'Rows clipped to the [{CFG.score_floor:.0f}, {CFG.score_cap:.0f}] display range: '
      f'{int((unclipped != train_scores).sum())} of {len(train_scores)}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(train_scores, bins=30, color='#4C8C8C', ax=ax)
ax.set_xlabel('Scorecard points')
ax.set_title('Distribution of applicant scores (train set)')
plt.tight_layout()
plt.show()

## 6. Challenger: LightGBM with fixed regularisation

**No hyperparameter search.** The previous version ran 25 Optuna trials optimising mean CV
AUC. With ~45 bads per validation fold the standard error on fold AUC is around +/- 0.06,
so the maximum over 25 trials beats its own mean by roughly 0.04 AUC through selection
alone — the reported "best CV AUC" was mostly a selection artifact, and the search was
tuning to noise. Evidence it found nothing real: the parameters it settled on
(`n_estimators=500`, `learning_rate=0.016`, `num_leaves=7`) describe a heavily shrunk,
near-additive model, which is what the sample size implied before the search started.

The parameters below encode that conclusion directly — shallow trees, strong shrinkage,
heavy L2 — with monotonic constraints on duration (+), amount (+) and age (-) so the
challenger stays defensible rather than merely flexible.

In [ ]:
LGB_PARAMS = dict(
    n_estimators=400,
    learning_rate=0.02,
    num_leaves=7,
    max_depth=4,
    min_child_samples=30,     # ~4% of the training set: no leaf fit to a handful of rows
    subsample=0.9,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_lambda=5.0,
    random_state=CFG.random_state,
    verbose=-1,
)

MONOTONE_DIRECTION = {'duration_months': 1, 'credit_amount': 1, 'age': -1}


def to_lgb_frame(X_raw: pd.DataFrame, categories: dict = None):
    '''LightGBM-ready copy with stable categorical dtypes.

    Category levels are pinned to the training frame's so a level missing from a scoring
    batch cannot silently shift the encoding.
    '''
    out = X_raw.copy()
    cats = {}
    for col in [c for c in out.columns if c in ALL_CATEGORICAL]:
        out[col] = (out[col].astype('category') if categories is None
                    else pd.Categorical(out[col], categories=categories[col]))
        cats[col] = out[col].cat.categories
    return out, cats


def make_lgbm(columns) -> lgb.LGBMClassifier:
    return lgb.LGBMClassifier(
        **LGB_PARAMS,
        monotone_constraints=[MONOTONE_DIRECTION.get(c, 0) for c in columns])


X_train_lgb, LGB_CATEGORIES = to_lgb_frame(X_train)
X_test_lgb, _ = to_lgb_frame(X_test, LGB_CATEGORIES)

lgb_cv_aucs = cross_val_score(make_lgbm(X_train_lgb.columns), X_train_lgb, y_train,
                              cv=cv, scoring='roc_auc')
lgbm = make_lgbm(X_train_lgb.columns).fit(X_train_lgb, y_train)

print(f'Scorecard  repeated-CV AUC: {cv_aucs.mean():.3f} +/- {cv_aucs.std():.3f}')
print(f'LightGBM   repeated-CV AUC: {lgb_cv_aucs.mean():.3f} +/- {lgb_cv_aucs.std():.3f}')
print(f'(n = {len(cv_aucs)} fits each, {CFG.n_splits} folds x {CFG.n_repeats} repeats)')

### What excluding the prohibited bases costs

The number a credit committee will ask for. Both models are refit on the full attribute
set, including `personal_status_sex` and `foreign_worker`, and compared on the same folds.
Removing features cannot improve fit, so this delta is a cost — the question is only
whether it is one worth paying.

In [ ]:
X_train_full = df.loc[X_train.index, [c for c in df.columns if c != 'target']]
X_full_lgb, _ = to_lgb_frame(X_train_full)

sc_full = cross_val_score(make_scorecard_pipeline(ALL_NUMERIC, ALL_CATEGORICAL),
                          X_train_full, y_train, cv=cv, scoring='roc_auc')
lgb_full = cross_val_score(make_lgbm(X_full_lgb.columns), X_full_lgb, y_train,
                           cv=cv, scoring='roc_auc')

exclusion_cost = pd.DataFrame(
    {'with prohibited bases': [sc_full.mean(), lgb_full.mean()],
     'without (deployed)': [cv_aucs.mean(), lgb_cv_aucs.mean()]},
    index=['Scorecard', 'LightGBM'])
exclusion_cost['AUC given up'] = (exclusion_cost['with prohibited bases']
                                  - exclusion_cost['without (deployed)'])
exclusion_cost.round(4)

### Is the challenger actually better? Bootstrap CIs and a DeLong test

The previous version concluded LightGBM gave "a modest AUC improvement" from a single
point estimate on 250 rows. At that sample size the standard error on AUC is around
+/- 0.035, so a gap of a point or two is indistinguishable from zero. Two tests settle it:
bootstrap confidence intervals on each metric, and **DeLong's test** for the paired AUC
difference on the same applicants.

In [ ]:
def ks_statistic(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    idx = int(np.argmax(np.abs(tpr - fpr)))
    return float(np.abs(tpr - fpr)[idx]), float(thresholds[idx])


def bootstrap_ci(y_true, y_score, stat_fn, n_boot=None, alpha=0.05):
    '''Stratified bootstrap percentile interval for a ranking metric.'''
    n_boot = n_boot or CFG.n_boot
    rng = np.random.default_rng(CFG.random_state)
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    pos, neg = np.flatnonzero(y_true == 1), np.flatnonzero(y_true == 0)
    stats = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([rng.choice(pos, pos.size, replace=True),
                              rng.choice(neg, neg.size, replace=True)])
        stats[i] = stat_fn(y_true[idx], y_score[idx])
    return (float(np.percentile(stats, 100 * alpha / 2)),
            float(np.percentile(stats, 100 * (1 - alpha / 2))))


def _midrank(x):
    order = np.argsort(x)
    sorted_x = x[order]
    ranked = np.empty(len(x), dtype=float)
    i = 0
    while i < len(x):
        j = i
        while j < len(x) and sorted_x[j] == sorted_x[i]:
            j += 1
        ranked[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(len(x), dtype=float)
    out[order] = ranked
    return out


def delong_test(y_true, score_a, score_b):
    '''DeLong's test for two correlated ROC curves. Returns (auc_a, auc_b, p_value).

    Fast DeLong (Sun & Xu, 2014): AUC variance from midrank structural components, which
    accounts for both curves being scored on the same applicants — an unpaired test would
    overstate the uncertainty of the difference.
    '''
    y_true = np.asarray(y_true)
    order = np.argsort(-y_true, kind='mergesort')      # positives first
    m = int(y_true.sum())
    n = len(y_true) - m
    preds = np.vstack([np.asarray(score_a), np.asarray(score_b)])[:, order]

    tx = np.vstack([_midrank(r) for r in preds[:, :m]])
    ty = np.vstack([_midrank(r) for r in preds[:, m:]])
    tz = np.vstack([_midrank(r) for r in preds])

    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    cov = np.cov((tz[:, :m] - tx) / n) / m + np.cov(1.0 - (tz[:, m:] - ty) / m) / n

    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        return float(aucs[0]), float(aucs[1]), 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return float(aucs[0]), float(aucs[1]), float(2 * (1 - norm.cdf(abs(z))))


test_proba_sc = scorecard.proba(X_test)
test_proba_lgb = lgbm.predict_proba(X_test_lgb)[:, 1]

rows = []
for name, proba in [('Scorecard', test_proba_sc), ('LightGBM', test_proba_lgb)]:
    auc_lo, auc_hi = bootstrap_ci(y_test, proba, roc_auc_score)
    ks, _ = ks_statistic(y_test, proba)
    ks_lo, ks_hi = bootstrap_ci(y_test, proba, lambda a, b: ks_statistic(a, b)[0])
    rows.append({'model': name,
                 'test AUC': f'{roc_auc_score(y_test, proba):.3f} [{auc_lo:.3f}, {auc_hi:.3f}]',
                 'KS': f'{ks:.3f} [{ks_lo:.3f}, {ks_hi:.3f}]',
                 'Brier': round(brier_score_loss(y_test, proba), 4)})
print(pd.DataFrame(rows).to_string(index=False))

auc_sc, auc_lgb, p_value = delong_test(y_test, test_proba_sc, test_proba_lgb)
print(f'\nDeLong paired test: scorecard {auc_sc:.3f} vs LightGBM {auc_lgb:.3f}, '
      f'difference {auc_lgb - auc_sc:+.3f}, p = {p_value:.3f}')
print('->', 'no statistically detectable difference at this sample size' if p_value >= 0.05
      else 'difference is statistically detectable')

## 7. Decision policy: expected loss in money, not error counts

**This is the section that changes decisions.**

The previous policy minimised `5*FN + 1*FP` — unit-free error counts. That assumes a
missed default on a 500 DM loan costs exactly what one on an 18,000 DM loan costs. It does
not. `credit_amount` is exposure-at-default and it is already in the feature matrix:

```
approve a bad applicant  -> loss = credit_amount * LGD
decline a good applicant -> loss = credit_amount * annual_margin * (duration_months / 12)
```

Both sides scale with exposure and the decline side also scales with term, so the implied
cost ratio is **not constant across the book** — it varies per applicant. A single flat
threshold cannot be optimal against that, which is why the policy is banded by exposure.
Thresholds are selected on **out-of-fold training predictions** and applied to the test set
once; choosing them on the test set would fit the cutoff to the sample used to report
performance.

In [ ]:
def loss_at(y_true, proba, threshold, amount, duration, lgd=None, margin=None) -> np.ndarray:
    return expected_loss(y_true, proba, threshold, amount, duration,
                         lgd=CFG.lgd if lgd is None else lgd,
                         annual_margin=CFG.annual_margin if margin is None else margin)


def legacy_count_cost(y_true, proba, threshold) -> float:
    '''The old flat 5:1 count objective, kept only to reproduce the previous policy.'''
    tn, fp, fn, tp = confusion_matrix(
        y_true, (np.asarray(proba) >= threshold).astype(int)).ravel()
    return fn * CFG.legacy_cost_fn + fp * CFG.legacy_cost_fp


def optimal_threshold(loss_fn, grid=None):
    grid = np.linspace(0.02, 0.98, 97) if grid is None else grid
    losses = np.array([loss_fn(t) for t in grid])
    best = int(np.argmin(losses))
    return float(grid[best]), float(losses[best])

In [ ]:
# Out-of-fold probabilities on the training set — used to choose thresholds, never to report.
oof_proba = {
    'Scorecard': oof_proba_sc,
    'LightGBM': cross_val_predict(make_lgbm(X_train_lgb.columns), X_train_lgb, y_train,
                                  cv=cv_single, method='predict_proba')[:, 1],
}

amount_train = X_train['credit_amount'].to_numpy(dtype=float)
duration_train = X_train['duration_months'].to_numpy(dtype=float)
amount_test = X_test['credit_amount'].to_numpy(dtype=float)
duration_test = X_test['duration_months'].to_numpy(dtype=float)

policy_rows, global_thresholds = [], {}
for name, proba in oof_proba.items():
    t_money, _ = optimal_threshold(
        lambda t, p=proba: loss_at(y_train, p, t, amount_train, duration_train).sum())
    t_legacy, _ = optimal_threshold(lambda t, p=proba: legacy_count_cost(y_train, p, t))
    global_thresholds[name] = t_money

    test_p = test_proba_sc if name == 'Scorecard' else test_proba_lgb
    for label, thr in [('flat 0.50', 0.50), ('legacy 5:1 counts', t_legacy),
                       ('exposure-weighted', t_money)]:
        losses = loss_at(y_test, test_p, thr, amount_test, duration_test)
        policy_rows.append({
            'model': name, 'policy': label, 'threshold': round(thr, 2),
            'approval rate': f'{(test_p < thr).mean():.1%}',
            'test loss (DM)': f'{losses.sum():,.0f}',
            'per applicant': f'{losses.mean():,.0f}'})

print(pd.DataFrame(policy_rows).to_string(index=False))
print('\nThe legacy row is the old 0.17-style policy re-derived. Compare its money loss '
      'against the exposure-weighted row: that gap is what the wrong objective was costing.')

### Banded policy: one threshold per exposure band

Because the loss ratio varies with exposure, the optimum is not one number. Small loans
tolerate a much higher `P(default)` before declining is worthwhile; large loans do not.
Bands are cut on **training** exposure terciles, and each band's threshold is chosen on
out-of-fold predictions within that band.

In [ ]:
PRIMARY = 'Scorecard'
primary_oof = oof_proba[PRIMARY]
primary_test_proba = test_proba_sc

band_edges = np.unique(np.quantile(amount_train, np.linspace(0, 1, CFG.n_amount_bands + 1)))
inner_edges = band_edges[1:-1]
train_bands = np.clip(np.searchsorted(inner_edges, amount_train, side='right'),
                      0, len(inner_edges))

bands, band_rows = [], []
for b in range(len(inner_edges) + 1):
    m = train_bands == b
    lo = band_edges[b]
    hi = band_edges[b + 1] if b < len(inner_edges) else None
    if m.sum() < 30 or y_train.to_numpy()[m].sum() < 5:
        thr = global_thresholds[PRIMARY]          # too thin to fit its own cutoff
        note = 'fell back to global (band too thin)'
    else:
        thr, _ = optimal_threshold(lambda t: loss_at(
            y_train.to_numpy()[m], primary_oof[m], t,
            amount_train[m], duration_train[m]).sum())
        note = ''
    bands.append({'max_amount': None if hi is None else float(hi), 'threshold': float(thr)})
    band_rows.append({'exposure band (DM)': f'{lo:,.0f} - ' + ('max' if hi is None else f'{hi:,.0f}'),
                      'n (train)': int(m.sum()),
                      'bad rate': f'{y_train.to_numpy()[m].mean():.1%}',
                      'threshold': round(thr, 2), 'note': note})

policy = BandedPolicy(bands, global_thresholds[PRIMARY], band_column='credit_amount',
                      lgd=CFG.lgd, annual_margin=CFG.annual_margin)
print(pd.DataFrame(band_rows).to_string(index=False))

flat_loss = loss_at(y_test, primary_test_proba, global_thresholds[PRIMARY],
                    amount_test, duration_test).sum()
banded_loss = loss_at(y_test, primary_test_proba, policy.threshold_for(amount_test),
                      amount_test, duration_test).sum()
print(f'\n{PRIMARY}, held-out test set:')
print(f'  flat exposure-weighted threshold : {flat_loss:>10,.0f} DM')
print(f'  banded by exposure               : {banded_loss:>10,.0f} DM  '
      f'({(banded_loss - flat_loss) / flat_loss:+.1%})')

### Sensitivity: the cutoff rests on two numbers we had to assume

`LGD` and `annual_margin` are not measurable from this dataset. Publishing a threshold
that depends on them without showing the dependence would be false confidence. If the
cutoff is stable across the grid the assumptions are not load-bearing; if it swings, the
honest deliverable is the grid rather than a single number.

In [ ]:
lgd_grid = [0.45, 0.60, 0.75, 0.90]
margin_grid = [0.03, 0.045, 0.06, 0.08]
sensitivity = pd.DataFrame(
    [[optimal_threshold(lambda t, l=l_, mg=m_: loss_at(
        y_train, primary_oof, t, amount_train, duration_train, lgd=l, margin=mg).sum())[0]
      for m_ in margin_grid] for l_ in lgd_grid],
    index=[f'LGD {v:.0%}' for v in lgd_grid],
    columns=[f'margin {v:.1%}' for v in margin_grid])
print('Cost-optimal threshold across loss assumptions:')
sensitivity.round(2)

## 8. Score bands and population stability

The band table turns a probability into an operating policy: it is where a credit officer
reads off the bad rate they are accepting at a given cutoff, and where a manual-review
band gets defined. **PSI** checks the score distribution has not shifted between
development and validation samples — the standard bar is < 0.10 stable, 0.10–0.25 watch,
> 0.25 investigate.

In [ ]:
score_edges = np.unique(np.quantile(train_scores, np.linspace(0, 1, CFG.n_score_bands + 1)))
score_edges[0], score_edges[-1] = -np.inf, np.inf

band_table = pd.DataFrame({
    'score_band': pd.cut(test_scores, bins=score_edges),
    'bad': y_test.to_numpy(), 'amount': amount_test,
}).groupby('score_band', observed=True).agg(
    n=('bad', 'size'), bads=('bad', 'sum'), bad_rate=('bad', 'mean'),
    avg_exposure=('amount', 'mean')).iloc[::-1]          # worst scores first
band_table['cum_pop'] = band_table['n'].cumsum() / band_table['n'].sum()
band_table['cum_bads_caught'] = band_table['bads'].cumsum() / band_table['bads'].sum()
band_table['lift'] = band_table['bad_rate'] / y_test.mean()
band_table.round(3)

In [ ]:
def psi(expected, actual, bins=10) -> float:
    edges = np.unique(np.percentile(expected, np.linspace(0, 100, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.clip(np.histogram(expected, edges)[0] / len(expected), 1e-4, None)
    a = np.clip(np.histogram(actual, edges)[0] / len(actual), 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))


score_psi = psi(train_scores, test_scores)
verdict = 'stable' if score_psi < 0.1 else 'watch' if score_psi < 0.25 else 'investigate'
print(f'PSI, train vs test scores: {score_psi:.4f} ({verdict})')

W_train, W_test = woe_step.transform(X_train), woe_step.transform(X_test)
feature_psi = pd.Series({c: psi(W_train[c], W_test[c], bins=5) for c in W_train.columns})
print('\nPSI by feature (train vs test WOE distribution):')
feature_psi.sort_values(ascending=False).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(band_table['cum_pop'], band_table['cum_bads_caught'], marker='o', color='#3B76AF')
ax.plot([0, 1], [0, 1], '--', color='grey', linewidth=1, label='random')
ax.set_xlabel('Share of applicants declined (worst scores first)')
ax.set_ylabel('Share of all defaults caught')
ax.set_title('Cumulative gains — scorecard')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Explainability and adverse-action reason codes

The scorecard is the deployed model and its per-feature decomposition is **exact** — the
points sum to the score by construction, with no sampling or approximation. That is a
stronger basis for an ECOA adverse-action notice than a SHAP attribution on a tree model,
which estimates a contribution rather than being one.

SHAP is still computed on the challenger, for the diagnostic it is genuinely good at:
showing where a flexible model would disagree with the additive one, and therefore what
the scorecard might be missing.

`PROTECTED_FEATURES` are not model inputs, so they cannot appear here. The filter inside
`ScorecardModel.reason_codes` is defence in depth, so a future feature-set change cannot
silently put a prohibited basis back onto a compliance document.

In [ ]:
scorecard.feature_labels = {
    'checking_status': 'Checking account status', 'duration_months': 'Loan duration (months)',
    'credit_history': 'Credit history', 'purpose': 'Loan purpose',
    'credit_amount': 'Credit amount', 'savings_status': 'Savings account status',
    'employment_since': 'Employment tenure', 'installment_rate': 'Instalment rate (% of income)',
    'other_debtors': 'Other debtors / guarantors', 'residence_since': 'Years at residence',
    'property': 'Property owned', 'age': 'Age',
    'other_installment_plans': 'Other instalment plans', 'housing': 'Housing status',
    'existing_credits': 'Existing credits at this bank', 'job': 'Employment category',
    'num_dependents': 'Number of dependents', 'telephone': 'Registered telephone',
}

for i in [0, 5]:
    codes = scorecard.reason_codes(X_test, row=i, top_k=3)
    thr = float(policy.threshold_for(amount_test[i])[0])
    print(f'\nApplicant {i}: score {test_scores.iloc[i]:.0f}, '
          f'P(default) {test_proba_sc[i]:.1%}, threshold {thr:.2f} -> '
          f'{"DECLINED" if test_proba_sc[i] >= thr else "APPROVED"}')
    for r in codes['adverse_reasons']:
        print(f'   + {r["label"]}: {r["value"]}   ({r["impact"]:+.3f})')
    for r in codes['positive_factors']:
        print(f'   - {r["label"]}: {r["value"]}   ({r["impact"]:+.3f})')

In [ ]:
explainer = shap.TreeExplainer(lgbm)
shap_raw = explainer.shap_values(X_test_lgb)
sv = shap_raw[1] if isinstance(shap_raw, list) else shap_raw

shap.summary_plot(sv, X_test_lgb, show=False)
plt.tight_layout()
plt.show()

In [ ]:
scorecard_strength = scorecard.risk_contributions(X_test).abs().mean()
scorecard_strength.index = [c[:-4] for c in scorecard_strength.index]

agreement = pd.DataFrame({
    'shap_rank': pd.Series(np.abs(sv).mean(axis=0), index=X_test_lgb.columns).rank(ascending=False),
    'scorecard_rank': scorecard_strength.rank(ascending=False),
}).dropna().sort_values('shap_rank')
agreement['rank_diff'] = (agreement['shap_rank'] - agreement['scorecard_rank']).abs()
print('Spearman rank correlation, SHAP vs scorecard attribution: '
      f"{agreement['shap_rank'].corr(agreement['scorecard_rank'], method='spearman'):.3f}")
agreement

**Reading the agreement check:** both methods put `checking_status`, `duration_months`,
`savings_status` and `credit_history` near the top, which is the useful confirmation — the
flexible model is not leaning on something the additive one considers spurious. Where the
rankings diverge, the likely cause is interaction effects the scorecard cannot represent.
That divergence is documentation in its own right: it tells a reviewer exactly where the
two models would explain the same decline differently.

## 10. Fairness screen

Not a compliance determination — a screen, run on attributes the model is **not allowed to
use**, at the deployed banded policy rather than an arbitrary 0.50 cutoff. Excluding an
attribute does not guarantee parity, because correlated features can proxy for it, so the
measurement matters regardless of the exclusion.

In [ ]:
audit = audit_test.reset_index(drop=True).copy()
audit['p_bad'] = primary_test_proba
audit['threshold'] = policy.threshold_for(amount_test)
audit['approved'] = (audit['p_bad'] < audit['threshold']).astype(int)
audit['actual_bad'] = y_test.reset_index(drop=True)
audit['age_band'] = pd.cut(audit['age'], bins=[17, 25, 35, 45, 55, 100],
                           labels=['18-25', '26-35', '36-45', '46-55', '56+'])


def group_report(frame, by) -> pd.DataFrame:
    rep = frame.groupby(by, observed=True).agg(
        n=('approved', 'size'), approval_rate=('approved', 'mean'),
        actual_default_rate=('actual_bad', 'mean'))
    rep['four_fifths_ratio'] = rep['approval_rate'] / rep['approval_rate'].max()
    rep['flag'] = np.where(rep['four_fifths_ratio'] < 0.8, 'review', '')
    # A ratio computed on a handful of applicants is noise, not a disparity.
    rep.loc[rep['n'] < 30, 'flag'] = 'n too small to read'
    return rep.round(3)


for attribute in ['age_band'] + PROTECTED_FEATURES:
    print(f'\n=== Approval parity by {attribute} (banded policy) ===')
    print(group_report(audit, attribute).to_string())

**Reading the fairness screen:** younger applicants are approved at a lower rate, tracking
a real difference in observed default rate by age in this data — plausibly a legitimate
risk signal rather than bias, and exactly the pattern a fair-lending review expects to see
documented rather than discovered. The `personal_status_sex` and `foreign_worker`
breakdowns are measured on attributes the model never saw, so any disparity there is a
**proxy effect** through correlated features, which is the finding worth escalating.
Several subgroups are small enough in a 250-row test set that the four-fifths ratio is
noise; those are flagged rather than interpreted. A production review needs a larger
sample, a statistician, and counsel.

## 11. Model card

**Model.** WOE scorecard — supervised monotonic binning, IV selection at 0.02, logistic
regression with every coefficient sign-constrained negative, Platt-calibrated on
out-of-fold values. Deployed as the primary decision engine.

**Challenger.** LightGBM, fixed regularisation, monotonic constraints on duration, amount
and age. Logged alongside, not decisioning. Section 6's DeLong test is the reason: at this
sample size the AUC difference is not statistically detectable, so there is no performance
case for paying the interpretability cost. If a larger sample later shows a real gap, the
challenger is already built and instrumented.

**Features.** 18 attributes. `personal_status_sex` and `foreign_worker` excluded as
prohibited bases under ECOA / Reg B, with the AUC given up quantified in Section 6. `age`
is retained under a monotonic constraint and audited in Section 10; jurisdictions that
prohibit it need a one-line change to `PROTECTED_FEATURES`.

**Policy.** Exposure-banded thresholds on calibrated `P(default)`, chosen on out-of-fold
training predictions against a money-denominated loss (`EAD x LGD` against forgone
margin), never on the test set. LGD and margin are documented assumptions published with a
sensitivity grid.

**Known limitations.**
- **Dataset ceiling.** 1,000 rows from 1994. AUC in the high 0.70s is this dataset's
  ceiling, not a tuning failure. Metrics carry confidence intervals because the point
  estimates are not precise beyond two decimals.
- **Reject inference.** These are observed outcomes on a book that was already screened.
  A model trained on accepted applicants is biased on the population it will actually
  face. Not fixable here — parcelling and augmentation need the rejected applications,
  which this dataset does not contain.
- **No temporal validation.** No dates, so no out-of-time test and no drift measurement.
  The PSI in Section 8 compares two random splits, which is weaker than a vintage
  comparison and must not be read as evidence of stability over time.
- **Assumed loss parameters.** LGD and margin are conventional values, not measured.

**Monitoring on deployment.** Monthly PSI on the score distribution and per feature;
quarterly recalibration of the Platt parameters; approval-rate and four-fifths tracking by
age band and by the excluded attributes; alert when a score band's realised bad rate
departs from the band table above.

### Serialised artifact

One file, containing the objects the API actually calls. The previous version documented
a scorecard and shipped a LightGBM pickle — the model card described a model that was not
in production. Writing the deployed objects and their metrics from the same cell is what
stops that drift recurring: retraining rewrites the artifact and the metrics together.

In [ ]:
BUNDLE_PATH = 'credit_model_bundle.joblib'

bundle = {
    'schema_version': 2,
    'created': date.today().isoformat(),
    'primary': 'scorecard',

    'scorecard': scorecard,          # ScorecardModel: probability, points, reason codes
    'policy': policy,                # BandedPolicy: threshold per exposure band
    'challenger': lgbm,              # logged, not decisioning
    'challenger_categories': {c: list(v) for c, v in LGB_CATEGORIES.items()},

    'model_features': MODEL_FEATURES,
    'numeric_features': NUMERIC_FEATS,
    'categorical_features': CATEGORICAL_FEATS,
    'protected_features': PROTECTED_FEATURES,
    'score_bands': [float(e) for e in score_edges],

    'metrics': {
        'cv_auc_mean': float(cv_aucs.mean()),
        'cv_auc_std': float(cv_aucs.std()),
        'test_auc': float(auc_sc),
        'test_auc_ci': list(bootstrap_ci(y_test, test_proba_sc, roc_auc_score)),
        'test_ks': ks_statistic(y_test, test_proba_sc)[0],
        'test_brier': float(brier_score_loss(y_test, test_proba_sc)),
        'challenger_test_auc': float(auc_lgb),
        'delong_p_value': float(p_value),
        'psi_train_test': score_psi,
        'n_train': int(len(X_train)),
        'n_test': int(len(X_test)),
    },
    'config': asdict(CFG),
    'versions': {'sklearn': sklearn.__version__, 'lightgbm': lgb.__version__,
                 'pandas': pd.__version__, 'numpy': np.__version__},
}

joblib.dump(bundle, BUNDLE_PATH)
print(f'Wrote {BUNDLE_PATH}')
print(json.dumps(bundle['metrics'], indent=2, default=float))

### Serving check

The same call path `credit_engine.py` uses, exercised here on a held-out applicant. If
this cell and the running app ever disagree, the engine has drifted from the notebook.

In [ ]:
def score_applicant(applicant: dict) -> dict:
    '''Score one raw applicant dict exactly as the serving engine does.'''
    row = pd.DataFrame([applicant])[MODEL_FEATURES]
    p_bad = float(scorecard.proba(row)[0])
    threshold = float(policy.threshold_for(row['credit_amount'].to_numpy(dtype=float))[0])
    return {'p_bad': round(p_bad, 4),
            'credit_score': int(scorecard.points(row).iloc[0]),
            'threshold': round(threshold, 3),
            'decision': 'DECLINED' if p_bad >= threshold else 'APPROVED',
            **scorecard.reason_codes(row, row=0, top_k=3)}


result = score_applicant(X_test.iloc[0].to_dict())
print(json.dumps({k: v for k, v in result.items()
                  if k not in ('adverse_reasons', 'positive_factors')}, indent=2))
for r in result['adverse_reasons']:
    print(f'   + {r["label"]}: {r["value"]}')